# Train M1 on a Colab T4

Runtime → Change runtime type → **T4 GPU**. Everything long-running writes to Google Drive, so a session that ends mid-way resumes where it stopped.

Order of work (see `docs/RUNBOOK.md`): the gold set is labeled and frozen, silver labels are committed, and `jpe/validate.py`, `jpe/score.py` and `jpe/train_config.py` are written and `configs/train.yaml` has no nulls left.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RUNS = "/content/drive/MyDrive/cprojectllm/runs"

In [ ]:
!git clone -q https://github.com/yougijain/cprojectllm.git
%cd cprojectllm
!pip install -q -r requirements-train.txt pytest

Post text is not in the repository, only IDs. Rebuild it from the HN API (a couple of minutes).

In [ ]:
!python scripts/fetch_posts.py --from-ids

The hand-written core has to pass its own tests before anything trains. Skips here mean something is still unwritten.

In [ ]:
!python -m pytest -q tests/test_train_config.py tests/test_validate.py tests/test_score.py

## Baselines on the same GPU

B2 (zero-shot) and B3 (few-shot) are the untuned student. The fp16 numbers here are the reference the 4-bit build is compared against.

In [ ]:
!python scripts/run_model.py B2 --split dev && python scripts/run_model.py B2 --split test
!python scripts/run_model.py B3 --split dev && python scripts/run_model.py B3 --split test

## Learning curve, then the final configuration with three seeds

Choose every value in `configs/train.yaml` on the dev split first (the dev loss and dev field accuracy are what you look at). Then:

In [ ]:
!python scripts/learning_curve.py --sizes 250 500 1000 2000 all --seeds 0 --runs-dir $RUNS

In [ ]:
!python scripts/learning_curve.py --sizes all --seeds 1 2 --runs-dir $RUNS

## Keep the results

Copy the predictions and training logs to Drive, then pull them into the local checkout and commit them there.

In [ ]:
!mkdir -p /content/drive/MyDrive/cprojectllm
!cp -r results /content/drive/MyDrive/cprojectllm/
!for d in $RUNS/*/; do mkdir -p /content/drive/MyDrive/cprojectllm/logs/$(basename $d); cp $d/train_log.json $d/loss.png /content/drive/MyDrive/cprojectllm/logs/$(basename $d)/ 2>/dev/null; done